# RTT routing, CONGA, and ECMP: figure export

This notebook prepares the 10 Gb/s, five-second CDF comparisons for the project README
and the detailed results notebook. It compares **ECMP, CONGA, Weighted RTT with a
fixed timeout, Weighted RTT with an RTT-derived timeout, and Random-Two with an
RTT-derived timeout** on the same input flows. ENT is the default; DM uses the same code.

Start by editing the configuration cell. Symmetric loads are 0.1–0.9; asymmetric
loads are 0.1–0.7. Every selected run must be present and every input flow must pass
the payload and independent sink checks before performance figures are exported.
If validation fails, diagnostic CSVs are saved and the last successful export is
left selected. Reduce the configured load range explicitly for a smaller comparison.

Run all cells, then open **rtt_cdf_10gbps_5s_results.ipynb** to read the complete
figure set with explanations. Each export is saved under
**readme_figures/rtt_cdf_10gbps_5s/WORKLOAD/TIMESTAMP/**. The workload's
**latest.json** identifies the most recent successful export. Separate snapshots
keep figures from different selections from being mixed.

The existing available-results notebook remains useful for monitoring partial
sweeps. This notebook reads saved simulator outputs and never starts a simulation.
Dependencies: Python 3, NumPy, Matplotlib, IPython, and the adjacent
**compare_checked.py** validation helper.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv
import io
import json
import math
import hashlib
import platform
import os
import sys
from html import escape

# Works when Jupyter starts in the ns-3 root or in this notebook's directory.
NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "compare_checked.py").is_file():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "8_hosts_v9(prob1)"
if not (NOTEBOOK_DIR / "compare_checked.py").is_file():
    raise RuntimeError("Open Jupyter from the ns-3 root or 8_hosts_v9(prob1).")
SIM_ROOT = NOTEBOOK_DIR.parent
os.environ["MPLCONFIGDIR"] = str(NOTEBOOK_DIR / ".matplotlib")
sys.path.insert(0, str(NOTEBOOK_DIR))

import numpy as np
import compare_checked as checked
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import HTML, Markdown, display

# compare_checked also supports command-line use and selects a file backend.
# Restore inline display for this notebook, without changing that helper file.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

# -------------------------- User configuration --------------------------
WORKLOAD = "ENT"                         # "ENT" or "DM"
SCENARIOS = ["sym", "asym"]              # Or just ["sym"] / ["asym"]
LOADS = {"sym": list(range(1, 10)), "asym": list(range(1, 8))}
RTT_POLICIES = ["weighted_fixed", "weighted_adaptive", "random_two_adaptive"]
MAKE_PER_LOAD_FIGURES = True              # CDF, tail, and utilization at every load
SHOW_FIGURES = False                      # The results notebook displays all saved PNGs
PNG_DPI = 200                             # SVG copies are exported as well

SERVER_COUNT, LEAF_COUNT, SPINE_COUNT = 4, 2, 2
UTIL_START_S, UTIL_END_S = 0.0, 5.0
FABRIC_CAPACITY_GBPS = 10.0
ASYM_LINK = (0, 0)                        # Leaf 0 -> Spine 0
ASYM_CAPACITY_GBPS = 5.0
INPUT_DIR = SIM_ROOT / "cdf_traffic_10gbps_5s"
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
EXPORT_ROOT = NOTEBOOK_DIR / "readme_figures" / "rtt_cdf_10gbps_5s"
OUTPUT_DIR = None                         # Set to a fresh snapshot when exporting

# Fixed and adaptive Weighted RTT use different colors as well as line styles.
POLICIES = {
    "ecmp": dict(family="conga", suffix="ECMP", label="ECMP", color="#555555", marker="o", linestyle="-"),
    "conga": dict(family="conga", suffix="Conga", label="CONGA", color="#0072B2", marker="s", linestyle="-"),
    "weighted_fixed": dict(family="rtt", suffix="WEIGHTED_ECMP_timeout_true_RTT", label="Weighted RTT / fixed", color="#009E73", marker="^", linestyle="-"),
    "weighted_adaptive": dict(family="rtt", suffix="WEIGHTED_ECMP_timeout_false_RTT", label="Weighted RTT / adaptive", color="#8B5FC7", marker="D", linestyle="--"),
    "random_two_adaptive": dict(family="rtt", suffix="POWER_OF_2_RANDOM_timeout_false_RTT", label="Random-Two / adaptive", color="#D55E00", marker="v", linestyle="-."),
}
BASELINES = ["ecmp", "conga"]
ALGORITHMS = BASELINES + RTT_POLICIES
GROUPS = ["All flows", "Small (<100 KiB)", "Medium (100 KiB-1 MiB)", "Large (>=1 MiB)"]
assert WORKLOAD in ("ENT", "DM")
assert SCENARIOS and set(SCENARIOS) <= {"sym", "asym"}
assert RTT_POLICIES and set(RTT_POLICIES) <= set(POLICIES) - set(BASELINES)
assert len(ALGORITHMS) == len(set(ALGORITHMS))
assert UTIL_END_S > UTIL_START_S and min(FABRIC_CAPACITY_GBPS, ASYM_CAPACITY_GBPS) > 0
assert all(LOADS[s] and len(LOADS[s]) == len(set(LOADS[s])) and all(1 <= n <= 9 for n in LOADS[s]) for s in SCENARIOS)

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                    "axes.grid": True, "grid.alpha": 0.18, "axes.axisbelow": True,
                    "figure.facecolor": "white", "font.size": 10})
print("Workload:", WORKLOAD, "| Topologies:", ", ".join(SCENARIOS))
print("Export parent:", EXPORT_ROOT / WORKLOAD)


## Measurement definitions

- **Application FCT:** elapsed time from the input flow's scheduled start until its
  entire requested payload arrives, including connection setup and completion
  during the drain period. It is simulated time, not the laptop's wall-clock runtime.
- **Flow matching:** source, destination, destination port, then nearest start
  time, as in the existing notebook. Flow size checks the match; it never chooses it.
- **Validation:** received payload must equal the input size; flow identity, times,
  completion flags, and independent sink counters must agree. FlowMonitor IP bytes
  include protocol overhead and are not a payload-size check.
- **Comparison population:** every plotted policy uses the same verified flows
  within a topology/load/size class. This exporter requires that population to
  include every requested flow. It never drops unfinished flows silently.
- **Mean and p99:** arithmetic mean and NumPy's linear 0.99 quantile of per-flow
  application FCT. Each flow counts once. A p99 is a distribution percentile,
  not a confidence interval. Size classes are small <100 KiB, medium
  100 KiB–<1 MiB, and large >=1 MiB.
- **Normalization:** ratios of means or ratios of p99 values over the same flows,
  not means of per-flow ratios. Overall mean/p99 use CONGA as the denominator;
  size-class normalized figures use ECMP. Below one means lower FCT.
- **Timeouts:** fixed means a 500-us flowlet timeout; adaptive means RTT-derived.
  Neither means the TCP connection timeout. Random-Two uses only the adaptive case.

The selected experiments use two leaves, two spines, four servers per leaf,
10 Gb/s links, and a five-second arrival window. In the asymmetric fabric,
Leaf 0–Spine 0 is 5 Gb/s. Runners currently use a 30-second drain period.
These are declared experiment settings; the CSV report is not a complete record
of the command line used for a run. Manifest checks below independently verify
the generated inputs.

The figures describe one trace and one routing run per case. Repeated seeds and
larger topologies are required before making general performance claims.


In [ ]:
ROOT_MARKER = "=== Per Link Root Queue Disc Drop ==="
DEVICE_MARKER = "=== Per Link Queue Drop ==="
REASON_MARKER = "=== Root Queue Disc Drop Reasons ==="
UTIL_MARKER = "=== Per Link Utilization ==="


def report_path(scenario, load, algorithm):
    policy = POLICIES[algorithm]
    return SIM_ROOT / f"{policy['family']}_{scenario}_cdf_10gbps_5s" / f"{WORKLOAD}_load_{load}_{policy['suffix']}.csv"


def file_set(path):
    return [path, Path(str(path) + ".flows.csv"), Path(str(path) + ".flows.csv.audit.csv")]


def snapshot(paths):
    """Detect a report being replaced or appended to during a read."""
    return tuple((p.stat().st_ino, p.stat().st_size, p.stat().st_mtime_ns) for p in paths)


def report_sections(path):
    """Read the appended tables, preserving absent versus empty sections."""
    sections, current, lines = {}, None, []
    for line in path.read_text().splitlines():
        if line.startswith("==="):
            if current is not None:
                sections[current] = list(csv.DictReader(io.StringIO("\n".join(lines))))
            current, lines = line.strip(), []
        elif current is not None and line.strip():
            lines.append(line)
    if current is not None:
        sections[current] = list(csv.DictReader(io.StringIO("\n".join(lines))))
    return sections


def available_file_set(path):
    """A missing future run is different from a partly written output set."""
    paths = file_set(path)
    present = [p.is_file() for p in paths]
    if not any(present):
        return "pending", "No output files yet"
    missing = [p.name for p, exists in zip(paths, present) if not exists or p.stat().st_size == 0]
    if missing:
        return "incomplete", "Missing or empty: " + ", ".join(missing)
    try:
        before = snapshot(paths)
        with path.open("rb") as stream:
            stream.seek(max(0, path.stat().st_size - 16384))
            tail = stream.read().decode("utf-8")
        tail_lines = tail.splitlines()
        final = next((line for line in reversed(tail_lines) if line.strip()), "")
        columns = next(csv.reader([final]))
        if UTIL_MARKER not in tail or not tail.endswith("\n") or len(columns) != 5:
            return "incomplete", "Report has not reached a complete final utilization row"
        if not all(checked.integer(columns[i]) >= 0 for i in (0, 1, 3)):
            return "incomplete", "Invalid final utilization row"
        if not math.isfinite(float(columns[4])):
            return "incomplete", "Invalid final utilization value"
        if snapshot(paths) != before:
            return "changing", "Output files changed during discovery"
    except (OSError, ValueError, TypeError, ArithmeticError, UnicodeError) as error:
        return "incomplete", str(error)
    return "available", "Report and both delivery files are present; validation follows"


def csv_write(path, rows, fields=None):
    """Always write a header, including when an issues table is empty."""
    rows = list(rows)
    fields = list(dict.fromkeys([*(fields or []), *(key for row in rows for key in row)])) or ["message"]
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def show_table(rows, columns, limit=30):
    rows = list(rows)
    if not rows:
        display(Markdown("No entries."))
        return
    def cell(value):
        if isinstance(value, float):
            return "n/a" if not math.isfinite(value) else f"{value:.5g}"
        return str(value)
    header = "".join(f"<th>{escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(cell(r.get(c, '')))}</td>" for c in columns) + "</tr>" for r in rows[:limit])
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"))
    if len(rows) > limit:
        print(f"Showing {limit} of {len(rows)} rows; the CSV contains all rows.")


def issue_row(scenario, load, algorithm, category, detail, trace_index=""):
    return dict(scenario=scenario, workload=WORKLOAD, load=load, algorithm=algorithm,
                category=category, trace_index=trace_index, issue=detail)


def nonnegative(value):
    value = checked.integer(value)
    if value < 0:
        raise ValueError("Negative counter")
    return value


def queue_total(sections, marker, expected_rows=None):
    """Absent/malformed counters stay unavailable; a present empty event table is zero."""
    if marker not in sections:
        return np.nan, "Queue table is absent: " + marker
    rows = sections[marker]
    try:
        identities = [(nonnegative(r["NodeId"]), nonnegative(r["DeviceId"])) for r in rows]
        if len(set(identities)) != len(identities):
            raise ValueError("Duplicate node/device counter")
        if expected_rows is not None and len(rows) != expected_rows:
            raise ValueError(f"Expected {expected_rows} interface rows; found {len(rows)}")
        return sum(nonnegative(r["DroppedPackets"]) for r in rows), ""
    except (KeyError, ValueError, TypeError, ArithmeticError) as error:
        return np.nan, f"{marker}: {error}"


def uplink_info(address, scenario):
    """Leaf-facing .1 addresses identify uplinks independently of node numbering."""
    parts = address.split(".")
    if len(parts) != 4 or parts[:2] != ["10", "1"] or parts[3] != "1":
        return None
    link = int(parts[2]) - LEAF_COUNT - 1
    if not 0 <= link < LEAF_COUNT * SPINE_COUNT:
        return None
    leaf, spine = divmod(link, SPINE_COUNT)
    capacity = ASYM_CAPACITY_GBPS if scenario == "asym" and (leaf, spine) == ASYM_LINK else FABRIC_CAPACITY_GBPS
    return f"Leaf {leaf} -> Spine {spine}", capacity


def inspect_available_run(scenario, load, algorithm, flows):
    """Validate all input flows and retain diagnostic counts even when FCT fails."""
    path = report_path(scenario, load, algorithm)
    before = snapshot(file_set(path))
    rows, file_problems = checked.inspect_run(flows, path, SERVER_COUNT)
    records, parse_problems = checked.read_flowmonitor(path, SERVER_COUNT)
    sections = report_sections(path)
    if snapshot(file_set(path)) != before:
        raise RuntimeError("Files changed during analysis; skipped until the next notebook run")
    problems = [issue_row(scenario, load, algorithm, "file", p) for p in dict.fromkeys(file_problems + parse_problems)]
    for marker, table in list(sections.items()):
        if any(None in row or any(value is None for value in row.values()) for row in table):
            problems.append(issue_row(scenario, load, algorithm, "logging table", "Malformed CSV row in " + marker))
            # A malformed table is unavailable, never an empty zero-drop table.
            del sections[marker]
    prefix = dict(scenario=scenario, workload=WORKLOAD, load=load, algorithm=algorithm)
    for row in rows:
        row.update(prefix)
        if row["issues"]:
            problems.append(issue_row(scenario, load, algorithm, "flow", row["issues"], row["trace_index"]))

    by_id = {str(r["raw"]["FlowID"]): r["raw"] for r in records}
    matched_ids = {str(r["flowmonitor_id"]) for r in rows if r["matched"]}
    # Missing or reused matches make a total misleading. Do not double count.
    loss_valid = len(matched_ids) == len(flows) and all(r["candidates"] == 1 for r in rows)
    tx = lost = np.nan
    if loss_valid:
        try:
            tx = sum(nonnegative(by_id[i]["TxPackets"]) for i in matched_ids)
            lost = sum(nonnegative(by_id[i]["LostPackets"]) for i in matched_ids)
            if lost > tx or tx == 0:
                raise ValueError("Invalid lost/transmitted packet counters")
        except (KeyError, ValueError, TypeError, ArithmeticError) as error:
            tx = lost = np.nan
            problems.append(issue_row(scenario, load, algorithm, "loss", str(error)))
    else:
        problems.append(issue_row(scenario, load, algorithm, "loss", "Forward loss withheld: missing, ambiguous, or reused FlowMonitor matches"))

    root_count, error = queue_total(sections, ROOT_MARKER, 2 * (SERVER_COUNT * LEAF_COUNT + LEAF_COUNT * SPINE_COUNT))
    if error:
        problems.append(issue_row(scenario, load, algorithm, "queue", error))
    device_count, error = queue_total(sections, DEVICE_MARKER)
    if error:
        problems.append(issue_row(scenario, load, algorithm, "queue", error))

    utilization = []
    seen_links = set()
    duplicate_links = set()
    for raw in sections.get(UTIL_MARKER, []):
        try:
            info = uplink_info(raw["IP Address"], scenario)
            if info is None:
                continue
            link, capacity = info
            if link in seen_links:
                duplicate_links.add(link)
                raise ValueError("Duplicate utilization row for " + link)
            seen_links.add(link)
            byte_count = nonnegative(raw["Bytes"])
            value = 100 * byte_count * 8 / (capacity * 1e9 * (UTIL_END_S - UTIL_START_S))
            reported = float(raw["UtilisationPercent"])
            utilization.append(dict(**prefix, link=link, bytes=byte_count, capacity_gbps=capacity,
                                    utilization_percent=value, reported_utilization_percent=reported))
            if not math.isfinite(reported) or abs(value - reported) > 0.06:
                problems.append(issue_row(scenario, load, algorithm, "utilization", f"{link}: CSV reports {reported:g}%; calculated {value:.4g}% using {capacity:g} Gb/s and the configured window"))
        except (KeyError, ValueError, TypeError, ArithmeticError) as error:
            problems.append(issue_row(scenario, load, algorithm, "utilization", str(error)))
    utilization = [row for row in utilization if row["link"] not in duplicate_links]

    expected_links = {f"Leaf {leaf} -> Spine {spine}"
                      for leaf in range(LEAF_COUNT) for spine in range(SPINE_COUNT)}
    for link in sorted(expected_links - {r["link"] for r in utilization}):
        problems.append(issue_row(scenario, load, algorithm, "utilization",
                                  "No usable utilization counter for " + link))

    unknown = sum(not isinstance(r["received_bytes"], int) for r in rows)
    missing = sum(max(0, r["expected_bytes"] - r["received_bytes"]) for r in rows if isinstance(r["received_bytes"], int))
    received = sum(r["received_bytes"] for r in rows if isinstance(r["received_bytes"], int))
    counts = Counter(r["size_status"] for r in rows)
    summary = dict(**prefix, label=POLICIES[algorithm]["label"], input_flows=len(flows),
                   exact_flows=counts["exact"], verified_flows=sum(r["verified"] for r in rows),
                   undersized_flows=counts["undersized"], oversized_flows=counts["oversized"],
                   missing_flows=counts["missing"], invalid_flows=counts["invalid"],
                   flows_with_issues=sum(bool(r["issues"]) for r in rows), unknown_received_flows=unknown,
                   exact_percent=100 * counts["exact"] / len(flows),
                   missing_payload_bytes=missing if not unknown else np.nan,
                   known_missing_payload_bytes=missing,
                   forward_tx_packets=tx, forward_lost_packets=lost,
                   forward_loss_percent=100 * lost / tx if tx > 0 else np.nan,
                   root_queue_disc_drops=root_count, device_queue_drops=device_count,
                   root_drops_per_delivered_mib=root_count / (received / 1024**2) if received > 0 and not unknown else np.nan,
                   file_issue_count=len(file_problems) + len(parse_problems),
                   report=path.relative_to(SIM_ROOT).as_posix())
    # Do not use a partially readable file's apparently healthy rows for FCT.
    eligible = not file_problems and not parse_problems
    fcts = {r["trace_index"]: r["application_fct_s"] * 1000 for r in rows
            if eligible and r["verified"] and isinstance(r["application_fct_s"], (int, float))
            and math.isfinite(r["application_fct_s"]) and r["application_fct_s"] > 0}
    return dict(rows=rows, summary=summary, issues=problems, fcts=fcts, eligible=eligible,
                utilization=utilization, sections=sections)


In [ ]:
def group_for_size(size):
    return GROUPS[1] if size < 100 * 1024 else GROUPS[2] if size < 1024**2 else GROUPS[3]


def common_fct_metrics(scenario, load, flows, runs):
    """Compute paired metrics for all selected policies; retain excluded counts."""
    names = [a for a in ALGORITHMS if a in runs and runs[a]["eligible"]]
    cohort = dict(scenario=scenario, workload=WORKLOAD, load=load,
                  available_algorithms=";".join(runs), compared_algorithms=";".join(names),
                  input_flows=len(flows), shared_flows=0, excluded_flows=len(flows), status="withheld")
    if set(names) != set(ALGORITHMS):
        cohort["reason"] = "Needs every selected policy to be present and structurally valid"
        return cohort, [], {}
    indices = set.intersection(*(set(runs[a]["fcts"]) for a in names))
    cohort.update(shared_flows=len(indices), excluded_flows=len(flows) - len(indices),
                  status="complete cohort" if len(indices) == len(flows) else "diagnostic subset",
                  reason="" if indices else "No common verified completed flows")
    metrics, series = [], {}
    for group in GROUPS:
        chosen = sorted(f["index"] for f in flows if f["index"] in indices and
                        (group == GROUPS[0] or group_for_size(f["size"]) == group))
        arrays = {a: np.array([runs[a]["fcts"][i] for i in chosen], dtype=float) for a in names}
        series[group] = arrays
        if not chosen:
            continue
        for a, values in arrays.items():
            metrics.append(dict(scenario=scenario, workload=WORKLOAD, load=load, algorithm=a,
                                size_group=group, n=len(chosen), mean_fct_ms=float(values.mean()),
                                p99_fct_ms=float(np.quantile(values, 0.99)),
                                mean_relative_to_ecmp=float(values.mean() / arrays["ecmp"].mean()),
                                mean_relative_to_conga=float(values.mean() / arrays["conga"].mean()),
                                p99_relative_to_ecmp=float(np.quantile(values, .99) / np.quantile(arrays["ecmp"], .99)),
                                p99_relative_to_conga=float(np.quantile(values, .99) / np.quantile(arrays["conga"], .99)),
                                excluded_input_flows=cohort["excluded_flows"]))
    return cohort, metrics, series


In [ ]:
def fingerprint(path):
    """Identify a source without copying large simulation output files."""
    stat = path.stat()
    return dict(path=path.relative_to(SIM_ROOT).as_posix(),
                size_bytes=stat.st_size, modified_ns=stat.st_mtime_ns)


def sha256(path):
    """Hash exported assets and the small input/configuration files."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def preflight():
    """Require the selected experiment matrix before reading all flow records."""
    inventory, problems, sources = [], [], []
    for scenario in SCENARIOS:
        for load in sorted(LOADS[scenario]):
            for algorithm in ALGORITHMS:
                path = report_path(scenario, load, algorithm)
                state, detail = available_file_set(path)
                inventory.append(dict(scenario=scenario, workload=WORKLOAD, load=load,
                                      algorithm=algorithm, availability=state, detail=detail,
                                      report=path.relative_to(SIM_ROOT).as_posix()))
                if state != "available":
                    problems.append(issue_row(scenario, load, algorithm, "file availability", detail))
                else:
                    sources.extend(fingerprint(p) for p in file_set(path))
    for load in sorted({load for s in SCENARIOS for load in LOADS[s]}):
        trace = INPUT_DIR / f"{WORKLOAD}_load_{load}.csv"
        for path in [trace, Path(str(trace) + ".manifest.csv")]:
            if not path.is_file() or path.stat().st_size == 0:
                problems.append(issue_row("input", load, "input", "trace", f"Missing or empty {path.name}"))
            else:
                sources.append(dict(**fingerprint(path), sha256=sha256(path)))
    return inventory, problems, sources


def inspect_input(load):
    """Cross-check the trace against its generator manifest and configuration."""
    path = INPUT_DIR / f"{WORKLOAD}_load_{load}.csv"
    manifest_path = Path(str(path) + ".manifest.csv")
    before = snapshot([path, manifest_path])
    flows = checked.read_trace(path)
    with manifest_path.open(newline="") as stream:
        manifest_rows = list(csv.DictReader(stream))
    if len(manifest_rows) != 1:
        raise ValueError("Expected exactly one manifest record")
    manifest = manifest_rows[0]
    if manifest["CDF"] != WORKLOAD:
        raise ValueError("Manifest workload differs from the selected workload")
    checks = {
        "TargetLoad": load / 10, "ServerCount": SERVER_COUNT, "LeafCount": LEAF_COUNT,
        "SpineCount": SPINE_COUNT, "LinkCount": 1,
        "SpineLeafCapacityGbps": FABRIC_CAPACITY_GBPS,
        "LeafServerCapacityGbps": FABRIC_CAPACITY_GBPS,
        "LaunchStartS": UTIL_START_S, "LaunchEndS": UTIL_END_S,
        "GeneratedFlows": len(flows), "GeneratedBytes": sum(f["size"] for f in flows),
    }
    for key, expected in checks.items():
        if not math.isclose(float(manifest[key]), expected, rel_tol=1e-10, abs_tol=1e-9):
            raise ValueError(f"Manifest {key}={manifest[key]} differs from expected {expected}")
    if any(not UTIL_START_S <= f["start"] < UTIL_END_S for f in flows):
        raise ValueError("Flow start falls outside the configured launch window")
    if any(max(f["sn"], f["dn"]) >= SERVER_COUNT * LEAF_COUNT or
           f["sn"] // SERVER_COUNT == f["dn"] // SERVER_COUNT for f in flows):
        raise ValueError("Trace contains an invalid endpoint or a non-cross-leaf flow")
    duration = UTIL_END_S - UTIL_START_S
    healthy_capacity = LEAF_COUNT * SPINE_COUNT * FABRIC_CAPACITY_GBPS * 1e9
    bits_per_second = sum(f["size"] for f in flows) * 8 / duration
    if not math.isclose(float(manifest["RealizedAggregateBps"]), bits_per_second, rel_tol=1e-10):
        raise ValueError("Manifest realized payload rate differs from trace bytes")
    if not math.isclose(float(manifest["ExpectedAggregateBps"]), load / 10 * healthy_capacity, rel_tol=1e-10):
        raise ValueError("Manifest target rate differs from the healthy-fabric load definition")
    if snapshot([path, manifest_path]) != before:
        raise ValueError("Input or manifest changed during analysis")
    sizes = np.array([f["size"] for f in flows], dtype=np.int64)
    summary = dict(workload=WORKLOAD, load=load, target_load=load / 10,
                   input_flows=len(flows), requested_bytes=int(sizes.sum()),
                   mean_size_bytes=float(sizes.mean()), p99_size_bytes=float(np.quantile(sizes, .99)),
                   realized_payload_gbps=bits_per_second / 1e9,
                   realized_load=bits_per_second / healthy_capacity,
                   traffic_seed=manifest["Seed"],
                   trace_sha256=sha256(path))
    return flows, summary, manifest


def collect_results(inventory):
    """Read all selected policies and record validation failures explicitly."""
    result = dict(inventory=inventory, issues=[], runs={}, summaries=[], metrics=[],
                  cohorts=[], series={}, inputs={}, workload_rows=[], manifests=[])
    for load in sorted({load for s in SCENARIOS for load in LOADS[s]}):
        try:
            flows, summary, manifest = inspect_input(load)
            result["inputs"][load] = flows
            result["workload_rows"].append(summary)
            result["manifests"].append(dict(load=load, **manifest))
        except (OSError, ValueError, KeyError, TypeError, ArithmeticError) as error:
            result["issues"].append(issue_row("input", load, "input", "trace", str(error)))
    for scenario in SCENARIOS:
        for load in sorted(LOADS[scenario]):
            flows = result["inputs"].get(load)
            if flows is None:
                continue
            print(f"  Validating {scenario}, load {load / 10:g}")
            case_runs = {}
            for algorithm in ALGORITHMS:
                try:
                    run = inspect_available_run(scenario, load, algorithm, flows)
                except (OSError, ValueError, TypeError, KeyError, ArithmeticError, RuntimeError) as error:
                    result["issues"].append(issue_row(scenario, load, algorithm, "file", str(error)))
                    continue
                result["runs"][scenario, load, algorithm] = run
                case_runs[algorithm] = run
                result["summaries"].append(run["summary"])
                result["issues"].extend(run["issues"])
            cohort, metrics, series = common_fct_metrics(scenario, load, flows, case_runs)
            result["cohorts"].append(cohort)
            result["metrics"].extend(metrics)
            result["series"][scenario, load] = series
    return result


def export_tables(result, directory):
    """Save evidence for valid and invalid runs before applying the figure gate."""
    for name, key, fields in [
        ("availability", "inventory", ["scenario", "load", "algorithm", "availability"]),
        ("issues", "issues", ["scenario", "workload", "load", "algorithm", "category", "trace_index", "issue"]),
        ("run_summary", "summaries", ["scenario", "load", "algorithm"]),
        ("cohorts", "cohorts", ["scenario", "load", "shared_flows", "excluded_flows"]),
        ("fct_summary", "metrics", ["scenario", "load", "algorithm", "size_group", "n"]),
        ("workload_summary", "workload_rows", ["workload", "load", "input_flows"]),
        ("input_manifests", "manifests", ["load"]),
    ]:
        csv_write(directory / f"{name}.csv", result[key], fields)
    for scenario in SCENARIOS:
        selected = [(key, run) for key, run in result["runs"].items() if key[0] == scenario]
        rows = [row for _, run in selected for row in run["rows"]]
        fields = list(rows[0]) if rows else ["trace_index", "issues"]
        csv_write(directory / scenario / "summary/flow_checks.csv", rows, fields)
        csv_write(directory / scenario / "summary/flow_issues.csv", [r for r in rows if r["issues"]], fields)
        csv_write(directory / scenario / "summary/uplink_utilization.csv",
                  [r for _, run in selected for r in run["utilization"]],
                  ["scenario", "workload", "load", "algorithm", "link", "bytes", "capacity_gbps",
                   "utilization_percent", "reported_utilization_percent"])
        for marker, name, fields in [
            (ROOT_MARKER, "root_queue_disc_drops",
             ["NodeId", "DeviceId", "IP Address", "QueueDiscType", "DroppedPackets",
              "DroppedBeforeEnqueue", "DroppedAfterDequeue", "DroppedBytes"]),
            (DEVICE_MARKER, "device_queue_drops", ["NodeId", "DeviceId", "IP Address", "DroppedPackets"]),
            (REASON_MARKER, "root_drop_reasons",
             ["NodeId", "DeviceId", "IP Address", "Stage", "Reason", "DroppedPackets"]),
        ]:
            table = [dict(scenario=scenario, workload=WORKLOAD, load=key[1], algorithm=key[2], **row)
                     for key, run in selected for row in run["sections"].get(marker, [])]
            csv_write(directory / scenario / "summary" / f"{name}.csv", table,
                      ["scenario", "workload", "load", "algorithm", *fields])


def require_valid_comparison(result):
    """Prevent a missing policy or a short flow from producing README FCT plots."""
    expected_cases = sum(len(LOADS[s]) for s in SCENARIOS)
    if len(result["runs"]) != expected_cases * len(ALGORITHMS):
        raise RuntimeError("Some selected runs could not be read; see issues.csv. No figures published.")
    if len(result["cohorts"]) != expected_cases or any(
        row["excluded_flows"] or row["status"] != "complete cohort" for row in result["cohorts"]
    ):
        raise RuntimeError("Some input flows failed validation; see cohorts.csv and flow_issues.csv. "
                           "No performance figures published.")
    if any(r["file_issue_count"] or r["verified_flows"] != r["input_flows"] for r in result["summaries"]):
        raise RuntimeError("A run did not verify every input flow; see run_summary.csv.")


## Figure definitions and plotting

The original color assignments and core panels are retained. Weighted RTT's fixed
and adaptive variants have different colors as well as different line styles.
Figures are exported as PNG for Markdown and SVG for later resizing.

The additional tail plots use the complementary CDF: the fraction of flows
whose FCT exceeds the horizontal value. Zero tail probabilities are omitted from
the logarithmic axis, never replaced by a small positive number. The ordinary
CDF is also exported for every selected load.

Forward FlowMonitor loss and queue drops are distinct measurements. Queue totals
cover both directions and the whole simulation, including drain; FCT and forward
loss concern input-matched forward connections. Their counters are not disjoint
sets of packets and must not be added. Zero recorded loss does not prove the
absence of reordering or retransmissions.

Utilization uses transmitted bytes during 0–5 s divided by each directed
link's physical capacity. All heatmaps use the same color scale. The 5 Gb/s
link is recalculated from bytes for every policy; RTT's reported nominal-rate
percentage is retained in the CSV for comparison. Missing counters are shown
as unavailable, not zero.


In [ ]:
def style(a):
    return {k: POLICIES[a][k] for k in ("label", "color", "marker", "linestyle")}


# Collected only while making this snapshot; the gallery never globs old images.
FIGURES = []
FIGURE_NOTES = {
    "input_workload": {
        "title": "Input flow sizes and realized load",
        "caption": "Left: an empirical flow-size CDF, where each requested flow has equal weight. Middle: the cumulative fraction of requested payload bytes in flows up to each size; large flows dominate this byte-weighted view. Right: realized payload arrival load versus the target. Both topologies reuse the same trace at a given load. The denominator is the healthy fabric's aggregate directed uplink capacity (40 Gb/s in this setup), including for asymmetric runs. Five-second sampling of a heavy-tailed distribution can give a realized load above or below its target. These curves describe the input traces, not routing performance."
    },
    "payload_delivery_vs_load": {
        "title": "Application delivery and validation",
        "caption": "All requested flows are counted. The panels show exact-size deliveries, missing application bytes, and flows with any validation finding. Unknown receive counts are unavailable rather than zero. The figure gate also requires every independent sink audit and flow match to pass. The last two axes use a linear region around zero, followed by logarithmic scaling."
    },
    "overall_mean_fct_vs_load": {
        "title": "Overall mean application FCT",
        "caption": "The left panel shows the arithmetic mean over all input flows, in milliseconds on a logarithmic axis. The right divides each mean by CONGA's mean for the same flows; lower is better and one equals CONGA. Small and large transfers each contribute one FCT value. The overall mean is not an average of the size-class means. Compare within a topology; asymmetric experiments have less capacity."
    },
    "overall_p99_fct_vs_load": {
        "title": "Overall p99 application FCT",
        "caption": "The left panel is the sample 99th percentile over all flow sizes. The right is the ratio of that percentile to CONGA's percentile for the same input-flow cohort. A larger mean improvement can coexist with a worse p99. These are quantiles of one simulation's flow population, not confidence intervals across repeated runs."
    },
    "fct_vs_load": {
        "title": "Mean and p99 FCT by flow size",
        "caption": "Columns split flows at 100 KiB and 1 MiB; rows show the arithmetic mean and sample p99. The vertical axes are logarithmic and may have different scales. Inspect large-flow behavior separately from small-flow latency. The fct_summary.csv file records the sample count for each class; a p99 from a small class depends on very few flows."
    },
    "normalized_mean_fct_vs_load": {
        "title": "Size-class mean FCT relative to ECMP",
        "caption": "Each point is mean(policy FCT) / mean(ECMP FCT) in the same size class and matched cohort. Below one means a lower mean FCT than ECMP; above one means a higher mean. This is a ratio of means, not the mean of individual flow ratios. The denominator is ECMP, unlike the overall normalized panels."
    },
    "normalized_p99_fct_vs_load": {
        "title": "Size-class p99 FCT relative to ECMP",
        "caption": "Each point divides the policy's sample p99 by ECMP's sample p99 over the same size class and flow cohort. Below one means a lower tail percentile. A p99 ratio is not the 99th percentile of per-flow speedups, and it does not describe run-to-run uncertainty."
    },
    "forward_packet_loss_vs_load": {
        "title": "Forward TCP packet loss",
        "caption": "FlowMonitor lost packets divided by transmitted packets, multiplied by 100, for forward TCP connections matched uniquely to the input flows. This includes connection-control traffic on those connections; it is not a fraction of missing payload bytes or a count of all retransmissions. Zero means no recorded forward losses under this counter."
    },
    "packet_loss_and_queue_drops": {
        "title": "Packet loss and queue drops",
        "caption": "The panels show forward FlowMonitor loss, root queue-disc drops across all interfaces and directions, root drops per delivered MiB, and device-queue drops. These counters have different scopes and may refer to overlapping packet events; do not add them. Root FqCoDel drops may represent queue management, not merely a physically full device queue. Event-stage and reason tables are exported under each topology's summary folder. Coincident zero-valued curves can cover one another."
    },
    "fct_cdf": {
        "title": "Per-load FCT distribution",
        "caption": "Within each size class, the vertical value is the fraction of verified completed flows with application FCT at or below the horizontal value. The horizontal scale is logarithmic; a curve further left reaches a given completion fraction sooner. Step shapes, especially for a small large-flow population, are expected. Each panel gives its number of flows per policy; no sizes are pooled across classes."
    },
    "fct_tail": {
        "title": "Per-load FCT tail",
        "caption": "The complementary CDF shows the fraction with application FCT strictly greater than the horizontal value, using logarithmic axes. At a fixed threshold, lower means fewer flows exceed it. The dotted 0.01 reference marks a one-percent tail. Ties are counted together and the zero-probability endpoint is omitted; a curve ending before another is not an incomplete transfer. Sample counts are shown, and the ordinary CDF remains available alongside this view."
    },
    "uplink_utilization": {
        "title": "Per-load directed uplink utilization",
        "caption": "Each cell is bytes transmitted on that leaf-to-spine link during 0–5 s, divided by that individual link's capacity over five seconds. Cells are not shares of a single total and do not sum to 100%. The reduced Leaf 0–Spine 0 link uses 5 Gb/s for asymmetric runs. Other links use 10 Gb/s. The same color scale is used across all exported heatmaps. Traffic during drain is excluded; these averages do not reveal instantaneous queueing or prove packet reordering."
    }
}


def save_figure(fig, directory, name):
    """Write both formats and add their relative paths to the figure index."""
    directory.mkdir(parents=True, exist_ok=True)
    png, svg = directory / f"{name}.png", directory / f"{name}.svg"
    fig.savefig(png, dpi=PNG_DPI, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    relative = directory.relative_to(OUTPUT_DIR)
    parts = relative.parts
    scenario = parts[0] if parts and parts[0] in ("sym", "asym") else "workload"
    load = int(parts[-1][5:]) if parts and parts[-1].startswith("load_") else None
    FIGURES.append(dict(kind=name, scenario=scenario, load=load,
                        title=FIGURE_NOTES[name]["title"], caption=FIGURE_NOTES[name]["caption"],
                        png=png.relative_to(OUTPUT_DIR).as_posix(),
                        svg=svg.relative_to(OUTPUT_DIR).as_posix(),
                        png_sha256=sha256(png), svg_sha256=sha256(svg),
                        recommended_for_readme=name in {
                            "overall_mean_fct_vs_load", "fct_vs_load",
                            "normalized_mean_fct_vs_load", "forward_packet_loss_vs_load"}))
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)


def finish_figure(fig, axes, directory, name, title):
    handles, labels = {}, {}
    for ax in np.asarray(axes, dtype=object).flat:
        hs, ls = ax.get_legend_handles_labels()
        for h, label in zip(hs, ls):
            handles[label], labels[label] = h, label
    if handles:
        # One-row figures need more space below their x-axis labels. Their
        # legends otherwise overlap the labels in the saved PNG/SVG exports.
        legend_y = -0.20 if np.asarray(axes, dtype=object).ndim == 1 else -0.08
        fig.legend(list(handles.values()), list(labels.values()), loc="lower center",
                   bbox_to_anchor=(0.5, legend_y), ncol=3, fontsize=8)
    fig.suptitle(title, weight="bold")
    save_figure(fig, directory, name)


def draw_series(ax, rows, loads, field):
    """NaN at missing loads prevents a line from bridging absent results."""
    for a in ALGORITHMS:
        indexed = {r["load"]: r for r in rows if r["algorithm"] == a}
        values = [indexed.get(load, {}).get(field, np.nan) for load in loads]
        if any(math.isfinite(v) for v in values):
            ax.plot(np.array(loads) / 10, values, **style(a), linewidth=1.8)
    if not any(math.isfinite(float(r.get(field, np.nan))) for r in rows):
        ax.text(.5, .5, "Counter/statistic unavailable", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("Target offered load")
    ax.set_xticks(np.array(loads) / 10)


def plot_scenario(result, scenario, directory):
    summaries = [r for r in result["summaries"] if r["scenario"] == scenario]
    if not summaries:
        return
    # Include intervening absent loads as gaps, but no empty future-load tail.
    loads = list(range(min(r["load"] for r in summaries), max(r["load"] for r in summaries) + 1))
    metrics = [r for r in result["metrics"] if r["scenario"] == scenario]
    diagnostic = any(r["excluded_flows"] for r in result["cohorts"] if r["scenario"] == scenario)
    title = f"{scenario.upper()} / {WORKLOAD}"
    fct_title = title + (" / diagnostic: see excluded flows in cohorts.csv" if diagnostic else " / common verified flows")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
    for ax, field, label in zip(axes, ["exact_percent", "missing_payload_bytes", "flows_with_issues"],
                               ["Exact-size flows (%)", "Missing application bytes", "Flows with validation issues"]):
        draw_series(ax, summaries, loads, field)
        ax.set_ylabel(label)
    axes[0].set_ylim(0, 101)
    axes[1].set_yscale("symlog", linthresh=1)
    axes[2].set_yscale("symlog", linthresh=1)
    finish_figure(fig, axes, directory, "payload_delivery_vs_load", title + ": delivery checks")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
    fields = [("forward_loss_percent", "Forward TCP loss (%)"), ("root_queue_disc_drops", "Root queue-disc drops"),
              ("root_drops_per_delivered_mib", "Root drops / delivered MiB"), ("device_queue_drops", "Device queue drops")]
    for ax, (field, label) in zip(axes.flat, fields):
        draw_series(ax, summaries, loads, field)
        ax.set_ylabel(label)
        ax.set_ylim(bottom=0)
    finish_figure(fig, axes, directory, "packet_loss_and_queue_drops", title + ": loss and queues")
    if not metrics:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")
    all_rows = [r for r in metrics if r["size_group"] == GROUPS[0]]
    draw_series(axes[0], all_rows, loads, "mean_fct_ms")
    draw_series(axes[1], all_rows, loads, "mean_relative_to_conga")
    axes[0].set(ylabel="Mean application FCT (ms)", title="Overall mean FCT", yscale="log")
    axes[1].set(ylabel="Mean FCT / CONGA mean FCT", title="Overall mean normalized to CONGA")
    axes[1].axhline(1, color="#222222", linestyle=":", linewidth=1)
    axes[1].set_ylim(bottom=0)
    finish_figure(fig, axes, directory, "overall_mean_fct_vs_load", fct_title)

    fig, axes = plt.subplots(2, 3, figsize=(15, 7), layout="constrained")
    for col, group in enumerate(GROUPS[1:]):
        rows = [r for r in metrics if r["size_group"] == group]
        for ax, field, label in [(axes[0, col], "mean_fct_ms", "Mean application FCT (ms)"),
                                 (axes[1, col], "p99_fct_ms", "p99 application FCT (ms)")]:
            draw_series(ax, rows, loads, field)
            ax.set(ylabel=label, yscale="log")
        axes[0, col].set_title(group)
    finish_figure(fig, axes, directory, "fct_vs_load", fct_title)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
    for ax, group in zip(axes, GROUPS[1:]):
        draw_series(ax, [r for r in metrics if r["size_group"] == group], loads, "mean_relative_to_ecmp")
        ax.axhline(1, color="#222222", linestyle=":", linewidth=1)
        ax.set(title=group, ylabel="Mean FCT / ECMP mean FCT", ylim=(0, None))
    finish_figure(fig, axes, directory, "normalized_mean_fct_vs_load", fct_title)


def plot_case(result, scenario, load, directory):
    groups = result["series"].get((scenario, load), {})
    cohort = next((r for r in result["cohorts"] if r["scenario"] == scenario and r["load"] == load), {})
    title = f"{scenario.upper()} / {WORKLOAD} / load {load / 10:g}"
    if any(len(values) for arrays in groups.values() for values in arrays.values()):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
        for ax, group in zip(axes, GROUPS[1:]):
            arrays = groups.get(group, {})
            n = 0
            for a, values in arrays.items():
                n = len(values)
                if n:
                    ax.step(np.sort(values), np.arange(1, n + 1) / n, where="post",
                            **{k: v for k, v in style(a).items() if k != "marker"})
            ax.set(xlabel="Application FCT (ms)", ylabel="Cumulative fraction",
                   title=f"{group}\nn = {n:,} per policy", xscale="log", ylim=(0, 1.02))
        finish_figure(fig, axes, directory, "fct_cdf",
                      title + f" / {cohort.get('status', '')}; excluded {cohort.get('excluded_flows', 0):,} input flows")

    names = [a for a in ALGORITHMS if (scenario, load, a) in result["runs"]]
    links = [f"Leaf {leaf} -> Spine {spine}" for leaf in range(LEAF_COUNT) for spine in range(SPINE_COUNT)]
    matrix = np.full((len(links), len(names)), np.nan)
    for col, a in enumerate(names):
        for row in result["runs"][scenario, load, a]["utilization"]:
            matrix[links.index(row["link"]), col] = row["utilization_percent"]
    if names and np.isfinite(matrix).any():
        fig, ax = plt.subplots(figsize=(max(8, 2 * len(names)), 4.5), layout="constrained")
        cmap = plt.get_cmap("YlGnBu").copy()
        cmap.set_bad("#eeeeee")
        im = ax.imshow(np.ma.masked_invalid(matrix), cmap=cmap, vmin=0, vmax=UTIL_HEATMAP_MAX)
        ax.set(xticks=range(len(names)), xticklabels=[POLICIES[a]["label"] for a in names],
               yticks=range(len(links)), yticklabels=links, title="Leaf-to-spine utilization, 0-5 s")
        ax.tick_params(axis="x", rotation=25)
        for row in range(len(links)):
            for col in range(len(names)):
                value = matrix[row, col]
                ax.text(col, row, f"{value:.1f}%" if np.isfinite(value) else "n/a", ha="center", va="center",
                        color="white" if np.isfinite(value) and value / UTIL_HEATMAP_MAX > 0.65 else "black", fontsize=8)
        fig.colorbar(im, ax=ax, label="Individual link utilization (%)")
        fig.suptitle(title, weight="bold")
        save_figure(fig, directory, "uplink_utilization")


In [ ]:
def plot_workload(result, directory):
    """Keep the input distribution separate from completed-flow FCT."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
    loads = sorted(result["inputs"])
    colors = plt.get_cmap("viridis")(np.linspace(.05, .90, len(loads)))
    for load, color in zip(loads, colors):
        sizes = np.sort(np.array([f["size"] for f in result["inputs"][load]], dtype=np.int64))
        unique, counts = np.unique(sizes, return_counts=True)
        # Collapse ties so the step is the empirical cumulative probability.
        axes[0].step(unique, np.cumsum(counts) / len(sizes), where="post",
                     label=f"load {load / 10:g}", color=color)
        axes[1].step(unique, np.cumsum(unique * counts) / sizes.sum(), where="post",
                     label=f"load {load / 10:g}", color=color)
    for ax, ylabel in zip(axes[:2], ["Cumulative fraction of flows", "Cumulative fraction of payload bytes"]):
        ax.set(xscale="log", xlabel="Requested flow size (bytes)", ylabel=ylabel, ylim=(0, 1.02))
    rows = sorted(result["workload_rows"], key=lambda r: r["load"])
    target = [r["target_load"] for r in rows]
    axes[2].plot(target, [r["realized_load"] for r in rows], "o-", color="#0072B2", label="Realized")
    axes[2].plot(target, target, ":x", color="#555555", label="Target")
    axes[2].set(xlabel="Target offered load", ylabel="Realized payload load", xticks=target, ylim=(0, None))
    axes[2].legend()
    axes[0].set_title("Flow-weighted input CDF")
    axes[1].set_title("Byte-weighted input CDF")
    axes[2].set_title("Five-second realized load")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(.5, -.17), ncol=5, fontsize=8)
    fig.suptitle(f"{WORKLOAD}: input traffic, shared between topologies", weight="bold")
    save_figure(fig, directory, "input_workload")


def plot_supplementary_scenario(result, scenario, directory):
    """Add tail and standalone loss figures without replacing the existing panels."""
    loads = sorted(LOADS[scenario])
    metrics = [r for r in result["metrics"] if r["scenario"] == scenario]
    title = f"{scenario.upper()} / {WORKLOAD} / all input flows verified"
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")
    rows = [r for r in metrics if r["size_group"] == GROUPS[0]]
    draw_series(axes[0], rows, loads, "p99_fct_ms")
    draw_series(axes[1], rows, loads, "p99_relative_to_conga")
    axes[0].set(ylabel="p99 application FCT (ms)", yscale="log", title="Overall p99 FCT")
    axes[1].set(ylabel="p99 FCT / CONGA p99 FCT", ylim=(0, None), title="Overall p99 normalized to CONGA")
    axes[1].axhline(1, color="#222222", linestyle=":", linewidth=1)
    finish_figure(fig, axes, directory, "overall_p99_fct_vs_load", title)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
    for ax, group in zip(axes, GROUPS[1:]):
        draw_series(ax, [r for r in metrics if r["size_group"] == group], loads, "p99_relative_to_ecmp")
        ax.axhline(1, color="#222222", linestyle=":", linewidth=1)
        ax.set(title=group, ylabel="p99 FCT / ECMP p99 FCT", ylim=(0, None))
    finish_figure(fig, axes, directory, "normalized_p99_fct_vs_load", title)

    fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
    draw_series(ax, [r for r in result["summaries"] if r["scenario"] == scenario],
                loads, "forward_loss_percent")
    ax.set_ylabel("Forward TCP packet loss (%)")
    ax.set_ylim(bottom=0)
    finish_figure(fig, [ax], directory, "forward_packet_loss_vs_load", f"{scenario.upper()} / {WORKLOAD}")


def plot_tail(result, scenario, load, directory):
    """Show empirical exceedance probabilities with explicit finite sample counts."""
    groups = result["series"][scenario, load]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
    for ax, group in zip(axes.flat, GROUPS):
        arrays = groups.get(group, {})
        n = 0
        for algorithm in ALGORITHMS:
            values = arrays.get(algorithm, np.array([]))
            n = len(values)
            if not n:
                continue
            x, counts = np.unique(values, return_counts=True)
            survival = (n - np.cumsum(counts)) / n
            if len(x) > 1:
                # Draw each nonzero interval all the way to the next distinct
                # FCT. Dropping the final zero-valued point from ax.step
                # would erase the last interval (or the entire two-flow curve).
                ax.stairs(survival[:-1], x, baseline=None,
                        **{k: v for k, v in style(algorithm).items() if k != "marker"})
        ax.axhline(.01, color="#888888", linestyle=":", linewidth=1)
        ax.set(xscale="log", yscale="log", xlabel="Application FCT (ms)",
               ylabel="Fraction with FCT > threshold", title=f"{group}\nn = {n:,} per policy",
               ylim=(min(1 / max(n, 1), .001), 1.1))
        if n < 2:
            ax.text(.5, .5, "Insufficient samples for a nonzero tail", transform=ax.transAxes,
                    ha="center", va="center")
    finish_figure(fig, axes, directory, "fct_tail",
                  f"{scenario.upper()} / {WORKLOAD} / load {load / 10:g}")


## Validate and export

The next cell checks the selected files, validates every flow, saves CSV summaries,
and exports the figures. It prints progress per topology/load and per plotting
stage. It does not overwrite an older snapshot. A failed attempt leaves its
diagnostics in its own folder and does not update **latest.json**.

A successful snapshot contains:

- **INDEX.md**: all figures with captions and links, plus a suggested README subset.
- **figure_manifest.json**: complete image inventory, checksums, and captions used by
  the reader notebook.
- **configuration.json**, **source_files.csv**, **input_manifests.csv**:
  declared settings and source provenance. Output-file size and modification time
  identify the files read; SHA-256 checksums are also saved for inputs and images.
- **availability.csv**, **issues.csv**, **run_summary.csv**, **cohorts.csv**,
  **fct_summary.csv**, **workload_summary.csv**: coverage, validation, and metrics.
- **workload/**: empirical input-distribution and realized-load figures.
- **sym/** and **asym/**: figures across loads and diagnostic CSVs in **summary/**.
- **sym/load_1/**, etc.: an FCT CDF, FCT tail, and link-utilization heatmap per load.

Utilization-reporting findings remain visible even when payload validation passes.
They do not stop FCT export. Other missing or malformed network counters appear
as unavailable, never as zero. Check these findings before using a loss or queue
figure. Exporting the gallery does not automatically change the project README.


In [ ]:
def export_comparison():
    """Create a complete snapshot and select it only after every export succeeds."""
    global OUTPUT_DIR, RUN_LABEL, UTIL_HEATMAP_MAX
    RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    OUTPUT_DIR = EXPORT_ROOT / WORKLOAD / RUN_LABEL
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    FIGURES.clear()
    print("Checking files; this snapshot is:", OUTPUT_DIR)
    inventory, preflight_issues, sources = preflight()
    csv_write(OUTPUT_DIR / "availability.csv", inventory)
    csv_write(OUTPUT_DIR / "issues.csv", preflight_issues,
              ["scenario", "workload", "load", "algorithm", "category", "trace_index", "issue"])
    if preflight_issues:
        show_table(preflight_issues, ["scenario", "load", "algorithm", "category", "issue"])
        raise RuntimeError("The selected experiment matrix is incomplete. See the saved diagnostics. "
                           "Complete those runs or explicitly select fewer loads; no figures published.")

    print(f"Reading and validating {len(inventory)} runs. This can take several minutes.")
    result = collect_results(inventory)
    # Cross-check the complete source set again before labeling the snapshot.
    changed = []
    for source in sources:
        try:
            current = fingerprint(SIM_ROOT / source["path"])
            if any(current[key] != source[key] for key in ("size_bytes", "modified_ns")):
                changed.append(source["path"])
        except OSError:
            changed.append(source["path"])
    for path in changed:
        result["issues"].append(issue_row("all", "", "input/output", "changing source",
                                         f"File changed since preflight: {path}"))
    csv_write(OUTPUT_DIR / "source_files.csv", sources,
              ["path", "size_bytes", "modified_ns", "sha256"])
    export_tables(result, OUTPUT_DIR)

    print("Validation findings by category:", dict(Counter(row["category"] for row in result["issues"])))
    show_table(result["cohorts"],
               ["scenario", "load", "input_flows", "shared_flows", "excluded_flows", "status"], limit=30)
    if result["issues"]:
        show_table(result["issues"], ["scenario", "load", "algorithm", "category", "issue"], limit=30)
    if changed:
        raise RuntimeError("Sources changed during analysis; the snapshot is withheld. Rerun after writers finish.")
    require_valid_comparison(result)

    utilization = [r["utilization_percent"] for run in result["runs"].values() for r in run["utilization"]]
    UTIL_HEATMAP_MAX = max([100.0, *utilization])
    configuration = dict(
        schema_version=1, workload=WORKLOAD, scenarios=SCENARIOS, loads=LOADS,
        algorithms=ALGORITHMS, policies={a: POLICIES[a] for a in ALGORITHMS},
        servers_per_leaf=SERVER_COUNT, leaves=LEAF_COUNT, spines=SPINE_COUNT,
        input_dir=INPUT_DIR.relative_to(SIM_ROOT).as_posix(), run_label_utc=RUN_LABEL,
        utilization_window_s=[UTIL_START_S, UTIL_END_S],
        fabric_capacity_gbps=FABRIC_CAPACITY_GBPS, asymmetric_link=list(ASYM_LINK),
        asymmetric_capacity_gbps=ASYM_CAPACITY_GBPS, heatmap_max_percent=UTIL_HEATMAP_MAX,
        fct_metric="application completion time from scheduled input start",
        cohort="every input flow, verified and shared by every selected policy",
        normalized_mean="ratio of arithmetic means over the same cohort",
        normalized_p99="ratio of linear sample 0.99 quantiles over the same cohort",
        size_boundaries_bytes=[100 * 1024, 1024**2],
        per_load_figures=MAKE_PER_LOAD_FIGURES, formats=["png", "svg"], png_dpi=PNG_DPI,
        python_version=platform.python_version(), numpy_version=np.__version__,
        matplotlib_version=plt.matplotlib.__version__,
        validation_helper_sha256=sha256(NOTEBOOK_DIR / "compare_checked.py"),
        settings_provenance="Declared analysis settings and input manifests; CSVs do not record the full run command.",
    )
    (OUTPUT_DIR / "configuration.json").write_text(json.dumps(configuration, indent=2) + "\n")

    print("Saving workload figures.")
    plot_workload(result, OUTPUT_DIR / "workload")
    for scenario in SCENARIOS:
        print(f"Saving {scenario} figures across loads.")
        directory = OUTPUT_DIR / scenario
        plot_scenario(result, scenario, directory)
        plot_supplementary_scenario(result, scenario, directory)
        if MAKE_PER_LOAD_FIGURES:
            for load in sorted(LOADS[scenario]):
                print(f"  {scenario}, load {load / 10:g}: CDF, tail, utilization")
                case_dir = directory / f"load_{load}"
                plot_case(result, scenario, load, case_dir)
                plot_tail(result, scenario, load, case_dir)

    if len({r["png"] for r in FIGURES}) != len(FIGURES):
        raise RuntimeError("Duplicate figure paths; no export selected.")
    validation = dict(
        selected_runs=len(inventory), analyzed_runs=len(result["runs"]),
        flow_records=sum(r["input_flows"] for r in result["summaries"]),
        all_input_flows_verified=True,
        issue_counts=dict(Counter(r["category"] for r in result["issues"])),
    )
    manifest = dict(schema_version=1, workload=WORKLOAD, run_label_utc=RUN_LABEL,
                    configuration="configuration.json", validation=validation, figures=FIGURES)
    (OUTPUT_DIR / "figure_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
    lines = [
        f"# {WORKLOAD}: RTT, CONGA, and ECMP comparisons", "",
        f"Export: {RUN_LABEL}. All {validation['analyzed_runs']} selected runs passed the flow-validation gate.",
        "The total flow-record count includes one record per input flow per algorithm/topology run.", "",
        "See [configuration](configuration.json), [run validation](run_summary.csv), "
        "[findings](issues.csv), [FCT statistics](fct_summary.csv), and [cohorts](cohorts.csv).", "",
        "Figures describe individual simulation runs; no repeated-seed uncertainty intervals are claimed.",
        "Relative FCT below one means lower FCT than the named baseline. "
        "Link utilizations have separate denominators and do not sum to 100%.", "",
        "## Suggested figures for the README", "",
    ]
    for item in FIGURES:
        if item["recommended_for_readme"]:
            lines.append(f"- [{item['scenario'].upper()}: {item['title']}]({item['png']})")
    lines.extend(["", "## Full figure set", ""])
    for item in FIGURES:
        context = item["scenario"].upper()
        if item["load"] is not None:
            context += f" / load {item['load'] / 10:g}"
        lines.extend([f"### {context}: {item['title']}", "", item["caption"], "",
                      f"![{item['title']}]({item['png']})", "",
                      f"[SVG version]({item['svg']})", ""])
    (OUTPUT_DIR / "INDEX.md").write_text("\n".join(lines))
    # Atomic replacement means the reader always selects a finished snapshot.
    pointer = EXPORT_ROOT / WORKLOAD / "latest.json"
    temporary = pointer.with_suffix(".json.pending")
    temporary.write_text(json.dumps(dict(schema_version=1, workload=WORKLOAD,
                                         snapshot=RUN_LABEL, manifest="figure_manifest.json"), indent=2) + "\n")
    temporary.replace(pointer)
    print(f"Finished: {len(FIGURES)} figures, each in PNG and SVG.")
    print("Snapshot:", OUTPUT_DIR)
    print("Now run and save rtt_cdf_10gbps_5s_results.ipynb to embed the full illustrated report.")
    return result


result = export_comparison()
